In [2]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        
import kagglehub


/kaggle/input/datasets/zikriazzuri/karhutla/karhutla-2026_semua-provinsi_2026-10-16.csv


In [3]:
df = pd.read_csv('/kaggle/input/datasets/zikriazzuri/karhutla/karhutla-2026_semua-provinsi_2026-10-16.csv')

print(df.shape)
print(df.info())
print(df.describe())

(6, 45)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 45 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Provinsi                         6 non-null      object 
 1   Hotspot Rendah                   5 non-null      float64
 2   Hotspot Sedang                   6 non-null      int64  
 3   Hotspot Tinggi                   6 non-null      int64  
 4   Total Hotspot                    6 non-null      int64  
 5   Fire Spot                        6 non-null      int64  
 6   Lahan Terbakar Indikatif (Ha)    6 non-null      float64
 7   Kualitas Udara (PM 25)           6 non-null      object 
 8   Jarak Pandang Bandara (Km)       6 non-null      object 
 9   Lahan Terbakar Harian (Ha)       6 non-null      float64
 10  MD (L)                           5 non-null      object 
 11  MD (P)                           5 non-null      object 
 12  Total MD          

In [4]:
df_raw = df.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r'[^\w]+', '_', regex=True).str.strip('_'))
df['tanggal_laporan'] = pd.to_datetime('2026-10-16')

df.columns.tolist()

['provinsi',
 'hotspot_rendah',
 'hotspot_sedang',
 'hotspot_tinggi',
 'total_hotspot',
 'fire_spot',
 'lahan_terbakar_indikatif_ha',
 'kualitas_udara_pm_25',
 'jarak_pandang_bandara_km',
 'lahan_terbakar_harian_ha',
 'md_l',
 'md_p',
 'total_md',
 'kabkot_terdampak',
 'luas_lahan_ditangani_darat',
 'luas_lahan_yang_akan_dipadamkan',
 'personil_bpbd',
 'personil_pemerintah_pusat',
 'tni',
 'polri',
 'pemda',
 'relawan',
 'polisi_hutan',
 'manggala_agni',
 'mpa',
 'satpol_pp',
 'perusahaan',
 'total_personel',
 'patroli_unit',
 'patroli_sortie',
 'wb_unit',
 'wb_sortie',
 'jumlah_wb_harian',
 'volume_wb_ltr',
 'omc_unit',
 'omc_sortie',
 'total_bahan_semai_kg',
 'luas_lahan_ditangani_udara',
 'total_unit_udara_hari_ini',
 'rekomendasi',
 'kesimpulan',
 'unit_patroli_standby',
 'unit_wb_standby',
 'unit_omc_standby',
 'total_unit_standby',
 'tanggal_laporan']

In [5]:
df['provinsi'] = df['provinsi'].str.strip().replace({
    'Sumsel': 'Sumatera Selatan', 'Kalbar': 'Kalimantan Barat',
    'Kalteng': 'Kalimantan Tengah', 'Kalsel': 'Kalimantan Selatan'})

In [6]:
df.isna().sum()[lambda s: s > 0]

hotspot_rendah               1
md_l                         1
md_p                         1
total_md                     1
personil_pemerintah_pusat    2
pemda                        2
polisi_hutan                 1
mpa                          2
satpol_pp                    2
perusahaan                   1
dtype: int64

In [7]:
cek = df[['hotspot_rendah','hotspot_sedang','hotspot_tinggi']].fillna(0).sum(axis=1)
(cek == df['total_hotspot'])
df['hotspot_rendah'] = df['hotspot_rendah'].fillna(0)

In [8]:
kolom_angka = ['personil_pemerintah_pusat','relawan','polisi_hutan','mpa','jumlah_wb_harian']
for c in kolom_angka:
    print(c, df[c].unique())

personil_pemerintah_pusat ['20' nan '0' '-']
relawan ['-' '390' '1862' '2100' '1163' '2686']
polisi_hutan ['-' '1271' '150' nan '13']
mpa ['4059' '690' nan '-' '0']
jumlah_wb_harian ['0' '49 kali/ 196.000 L' '100x Drop/ 400.000 L' '84 kali / 336.000 L'
 '50x Drop/ 158.000 L']


In [9]:
for c in kolom_angka:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(',', '.').replace({'-': np.nan, 'nan': np.nan}),
        errors='coerce')
df[kolom_angka]

,personil_pemerintah_pusat,relawan,polisi_hutan,mpa,jumlah_wb_harian
0,20.0,NaN,NaN,4059.0,0.0
1,NaN,390.0,1271.0,690.0,NaN
2,0.0,1862.0,150.0,NaN,NaN
3,NaN,2100.0,NaN,NaN,NaN
4,NaN,1163.0,NaN,NaN,0.0
5,0.0,2686.0,13.0,0.0,NaN


In [10]:
wb = df_raw['jumlah WB harian'].astype(str).str.lower().str.strip()

df['wb_drop_harian'] = pd.to_numeric(wb.str.extract(r'(\d+)\s*(?:kali|x)')[0], errors='coerce')

df['wb_liter_teks'] = pd.to_numeric(
    wb.str.extract(r'([\d\.]+)\s*l')[0].str.replace('.', '', regex=False), errors='coerce')

df.loc[wb == '0', ['wb_drop_harian', 'wb_liter_teks']] = 0

df[['provinsi', 'jumlah_wb_harian', 'wb_drop_harian', 'wb_liter_teks', 'volume_wb_ltr']]

,provinsi,jumlah_wb_harian,wb_drop_harian,wb_liter_teks,volume_wb_ltr
0,Riau,0.0,0.0,0.0,0
1,Jambi,NaN,49.0,196000.0,196000
2,Sumatera Selatan,NaN,100.0,400000.0,400000
3,Kalimantan Barat,NaN,84.0,336000.0,336000
4,Kalimantan Tengah,0.0,0.0,0.0,0
5,Kalimantan Selatan,NaN,50.0,158000.0,158000


In [11]:
df = df.drop(columns='jumlah_wb_harian')
df = df.drop(columns='wb_liter_teks')

In [12]:
s = (df['kualitas_udara_pm_25'].str.lower()
       .str.replace('sanagat', 'sangat').str.strip())
parts = s.str.split('-', expand=True).apply(lambda c: c.str.strip())
skala = {'baik':1, 'sedang':2, 'tidak sehat':3, 'sangat tidak sehat':4, 'berbahaya':5}
df['udara_terbaik']  = parts[0].map(skala)
df['udara_terburuk'] = parts[1].fillna(parts[0]).map(skala)
df[['provinsi','kualitas_udara_pm_25','udara_terbaik','udara_terburuk']]

,provinsi,kualitas_udara_pm_25,udara_terbaik,udara_terburuk
0,Riau,Baik - Sangat Tidak Sehat,1,4
1,Jambi,Tidak sehat-berbahaya,3,5
2,Sumatera Selatan,Sedang - Berbahaya,2,5
3,Kalimantan Barat,Sanagat Tidak Sehat,4,4
4,Kalimantan Tengah,Tidak Sehat-Berbahaya,3,5
5,Kalimantan Selatan,Sangat Tidak Sehat - Berbahaya,4,5


In [13]:
nums = (df['jarak_pandang_bandara_km'].astype(str).str.replace(',', '.')
          .str.findall(r'\d+\.?\d*'))
df['jarak_pandang_min'] = nums.apply(lambda x: min(map(float, x)) if x else np.nan)
df['jarak_pandang_max'] = nums.apply(lambda x: max(map(float, x)) if x else np.nan)
df['jarak_pandang_batas'] = df['jarak_pandang_bandara_km'].astype(str).str.contains('[<≤]')
df[['provinsi','jarak_pandang_bandara_km','jarak_pandang_min','jarak_pandang_max','jarak_pandang_batas']]

,provinsi,jarak_pandang_bandara_km,jarak_pandang_min,jarak_pandang_max,jarak_pandang_batas
0,Riau,3.5,3.5,3.5,False
1,Jambi,1,1.0,1.0,False
2,Sumatera Selatan,≤ 0.7 - 2,0.7,2.0,True
3,Kalimantan Barat,0.2 - 7,0.2,7.0,False
4,Kalimantan Tengah,0.2,0.2,0.2,False
5,Kalimantan Selatan,< 7,7.0,7.0,True


In [14]:
df.loc[df['provinsi'] == 'Kalimantan Selatan', 'jarak_pandang_min'] = np.nan
df[['provinsi','jarak_pandang_bandara_km','jarak_pandang_min','jarak_pandang_max','jarak_pandang_batas']]

,provinsi,jarak_pandang_bandara_km,jarak_pandang_min,jarak_pandang_max,jarak_pandang_batas
0,Riau,3.5,3.5,3.5,False
1,Jambi,1,1.0,1.0,False
2,Sumatera Selatan,≤ 0.7 - 2,0.7,2.0,True
3,Kalimantan Barat,0.2 - 7,0.2,7.0,False
4,Kalimantan Tengah,0.2,0.2,0.2,False
5,Kalimantan Selatan,< 7,NaN,7.0,True


In [15]:
komp_personel = ['personil_bpbd','personil_pemerintah_pusat','tni','polri','pemda',
                 'relawan','polisi_hutan','manggala_agni','mpa','satpol_pp','perusahaan']
df['selisih_personel'] = df['total_personel'] - df[komp_personel].fillna(0).sum(axis=1)
df['selisih_standby'] = df['total_unit_standby'] - df[['unit_patroli_standby','unit_wb_standby','unit_omc_standby']].sum(axis=1)
df[['provinsi','selisih_personel','selisih_standby']]

,provinsi,selisih_personel,selisih_standby
0,Riau,0.0,0
1,Jambi,0.0,0
2,Sumatera Selatan,0.0,0
3,Kalimantan Barat,0.0,0
4,Kalimantan Tengah,0.0,0
5,Kalimantan Selatan,0.0,0


In [16]:
komp_personel = ['personil_bpbd','personil_pemerintah_pusat','tni','polri','pemda',
                 'relawan','polisi_hutan','manggala_agni','mpa','satpol_pp','perusahaan']
df[komp_personel] = df[komp_personel].fillna(0)

kolom_inti = ['provinsi','tanggal_laporan','hotspot_rendah','hotspot_sedang','hotspot_tinggi',
              'total_hotspot','fire_spot','lahan_terbakar_harian_ha','udara_terburuk',
              'jarak_pandang_min','kabkot_terdampak','total_personel','patroli_unit',
              'wb_unit','wb_drop_harian','volume_wb_ltr','omc_unit',
              'luas_lahan_ditangani_darat','luas_lahan_ditangani_udara',
              'total_unit_udara_hari_ini','total_unit_standby']
df_clean = df[kolom_inti].copy()

df_clean.info()
df_clean.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   provinsi                    6 non-null      object        
 1   tanggal_laporan             6 non-null      datetime64[ns]
 2   hotspot_rendah              6 non-null      float64       
 3   hotspot_sedang              6 non-null      int64         
 4   hotspot_tinggi              6 non-null      int64         
 5   total_hotspot               6 non-null      int64         
 6   fire_spot                   6 non-null      int64         
 7   lahan_terbakar_harian_ha    6 non-null      float64       
 8   udara_terburuk              6 non-null      int64         
 9   jarak_pandang_min           5 non-null      float64       
 10  kabkot_terdampak            6 non-null      int64         
 11  total_personel              6 non-null      int64         
 12

provinsi                      0
tanggal_laporan               0
hotspot_rendah                0
hotspot_sedang                0
hotspot_tinggi                0
total_hotspot                 0
fire_spot                     0
lahan_terbakar_harian_ha      0
udara_terburuk                0
jarak_pandang_min             1
kabkot_terdampak              0
total_personel                0
patroli_unit                  0
wb_unit                       0
wb_drop_harian                0
volume_wb_ltr                 0
omc_unit                      0
luas_lahan_ditangani_darat    0
luas_lahan_ditangani_udara    0
total_unit_udara_hari_ini     0
total_unit_standby            0
dtype: int64

In [17]:
m = df_clean.copy()
m['pct_hotspot_tinggi']         = m['hotspot_tinggi'] / m['total_hotspot'] * 100
m['personel_per_hotspot']       = m['total_personel'] / m['total_hotspot']
m['unit_udara_per_100_hotspot'] = m['total_unit_udara_hari_ini'] / m['total_hotspot'] * 100
m['rank_hotspot']               = m['total_hotspot'].rank(ascending=False).astype(int)
m['rank_personel_per_hotspot']  = m['personel_per_hotspot'].rank(ascending=False).astype(int)

m[['provinsi','total_hotspot','total_personel','pct_hotspot_tinggi',
   'personel_per_hotspot','unit_udara_per_100_hotspot',
   'rank_hotspot','rank_personel_per_hotspot']].round(2).sort_values('rank_hotspot')

,provinsi,total_hotspot,total_personel,pct_hotspot_tinggi,personel_per_hotspot,unit_udara_per_100_hotspot,rank_hotspot,rank_personel_per_hotspot
4,Kalimantan Tengah,4686,4686,7.21,1.00,0.19,1,6
2,Sumatera Selatan,1344,6615,5.95,4.92,0.52,2,5
5,Kalimantan Selatan,481,7268,6.24,15.11,1.46,3,4
3,Kalimantan Barat,466,14458,3.43,31.03,2.36,4,3
1,Jambi,137,5984,6.57,43.68,3.65,5,2
0,Riau,4,14837,0.00,3709.25,150.00,6,1


In [18]:
kolom_p = ['personil BPBD','personil pemerintah pusat','TNI','Polri','Pemda',
           'Relawan','Polisi Hutan','Manggala Agni','MPA','Satpol PP','Perusahaan','Total Personel']
df_raw.loc[df_raw['Provinsi'] == 'Kalteng', kolom_p].T


,4
personil BPBD,1182
personil pemerintah pusat,NaN
TNI,1403
Polri,762
Pemda,NaN
Relawan,1163
Polisi Hutan,NaN
Manggala Agni,176
MPA,NaN
Satpol PP,NaN


In [19]:
m['personel_per_fire_spot'] = m['total_personel'] / m['fire_spot']
m['personel_per_ha_harian'] = m['total_personel'] / m['lahan_terbakar_harian_ha']
m['pct_hotspot_jadi_api']   = m['fire_spot'] / m['total_hotspot'] * 100

print(m[['personel_per_fire_spot', 'personel_per_ha_harian', 'pct_hotspot_jadi_api']])

   personel_per_fire_spot  personel_per_ha_harian  pct_hotspot_jadi_api
0             7418.500000             5934.800000             50.000000
1              427.428571               47.872000             10.218978
2               77.823529               19.670522              6.324405
3             1445.800000              382.082452              2.145923
4               55.785714                7.877351              1.792574
5              279.538462              134.567673              5.405405


In [20]:
for _, r in df_raw.iterrows():
    print('=====', r['Provinsi'])
    print('REKOMENDASI:', r['rekomendasi'])
    print('KESIMPULAN :', r['kesimpulan'])

===== Riau
REKOMENDASI: Mohon dukungan dana operasional tambahan (uang semangat) untuk Kodam, Polda, BPBD guna menjaga moril satgas di lapangan yang baru didukung untuk 10 hari.
KESIMPULAN : Terbakar ± 2.5 Ha, berhasil dipadamkan tuntas 2.5 Ha (Sisa 0 Ha). Kualitas udara SANGAT TIDAK SEHAT di 4 wilayah murni asap kiriman. Heli Patroli PK-WSD operasional penuh.
===== Jambi
REKOMENDASI: 1. Memanfaatkan peralatan yang ada untuk pemadaman serta koordinasi dengan pihak perusahaan terdekat guna pembuatan kanal-kanal 
2. Memanfaatkan sortie penerbangan semaksimal mungkin untuk upaya pemadaman
3. Mengoptimalkan pemantauan kondisi awan dan perkembangan cuaca untuk menentukan waktu serta lokasi penyemaian yang paling tepat.  
4. Memprioritaskan wilayah yang memiliki firespot aktif atau berpotensi membesar dalam pelaksanaan OMC.  
5. `Memanfaatkan Pesawat Cessna Caravan yang sebelumnya digunakan OMC untuk melakukan patroli.  

KESIMPULAN : 1. Dari total lahan yang harus dipadamkan pada tanggal 14

In [21]:
p = df[['provinsi','lahan_terbakar_harian_ha','luas_lahan_yang_akan_dipadamkan',
        'luas_lahan_ditangani_darat','luas_lahan_ditangani_udara']].copy()

p['dipadamkan_ha']  = p['luas_lahan_ditangani_darat'] + p['luas_lahan_ditangani_udara']
p['sisa_hitung_ha'] = p['lahan_terbakar_harian_ha'] - p['dipadamkan_ha']
p['selisih_sisa']   = p['sisa_hitung_ha'] - p['luas_lahan_yang_akan_dipadamkan']
p['pct_tertangani'] = (p['dipadamkan_ha'] / p['lahan_terbakar_harian_ha'] * 100).round(1)
p['pct_udara']      = (p['luas_lahan_ditangani_udara'] / p['dipadamkan_ha'] * 100).round(1)
p

,provinsi,lahan_terbakar_harian_ha,luas_lahan_yang_akan_dipadamkan,luas_lahan_ditangani_darat,luas_lahan_ditangani_udara,dipadamkan_ha,sisa_hitung_ha,selisih_sisa,pct_tertangani,pct_udara
0,Riau,2.50,0.00,2.50,0.0,2.50,0.00,0.0,100.0,0.0
1,Jambi,125.00,93.50,28.00,3.5,31.50,93.50,0.0,25.2,11.1
2,Sumatera Selatan,336.29,56.00,257.29,23.0,280.29,56.00,0.0,83.3,8.2
3,Kalimantan Barat,37.84,8.34,21.50,8.0,29.50,8.34,0.0,78.0,27.1
4,Kalimantan Tengah,594.87,521.08,73.79,0.0,73.79,521.08,0.0,12.4,0.0
5,Kalimantan Selatan,54.01,614.00,43.41,4.0,47.41,6.60,-607.4,87.8,8.4


In [22]:
df_raw[['Provinsi','Lahan Terbakar Harian (Ha)','luas lahan yang akan dipadamkan',
        'luas lahan ditangani darat','luas lahan ditangani udara']]

,Provinsi,Lahan Terbakar Harian (Ha),luas lahan yang akan dipadamkan,luas lahan ditangani darat,luas lahan ditangani udara
0,Riau,2.50,0.00,2.50,0.0
1,Jambi,125.00,93.50,28.00,3.5
2,Sumsel,336.29,56.00,257.29,23.0
3,Kalbar,37.84,8.34,21.50,8.0
4,Kalteng,594.87,521.08,73.79,0.0
5,Kalsel,54.01,614.00,43.41,4.0


In [23]:
p['sisa_dilaporkan_ha'] = p['luas_lahan_yang_akan_dipadamkan']
p['sisa_ha'] = p['sisa_hitung_ha']                       # dipakai untuk analisis
p['flag_data'] = np.where(p['selisih_sisa'].abs() > 0.01,
                          'sisa dilaporkan tidak cocok; dipakai hasil hitung', '')

total_sisa = p['sisa_ha'].sum()
p['pct_dari_total_sisa'] = (p['sisa_ha'] / total_sisa * 100).round(1)

p[['provinsi','lahan_terbakar_harian_ha','luas_lahan_ditangani_darat','luas_lahan_ditangani_udara',
   'dipadamkan_ha','sisa_ha','pct_tertangani','pct_udara','pct_dari_total_sisa','flag_data']]

,provinsi,lahan_terbakar_harian_ha,luas_lahan_ditangani_darat,luas_lahan_ditangani_udara,dipadamkan_ha,sisa_ha,pct_tertangani,pct_udara,pct_dari_total_sisa,flag_data
0,Riau,2.50,2.50,0.0,2.50,0.00,100.0,0.0,0.0,
1,Jambi,125.00,28.00,3.5,31.50,93.50,25.2,11.1,13.6,
2,Sumatera Selatan,336.29,257.29,23.0,280.29,56.00,83.3,8.2,8.2,
3,Kalimantan Barat,37.84,21.50,8.0,29.50,8.34,78.0,27.1,1.2,
4,Kalimantan Tengah,594.87,73.79,0.0,73.79,521.08,12.4,0.0,76.0,
5,Kalimantan Selatan,54.01,43.41,4.0,47.41,6.60,87.8,8.4,1.0,sisa dilaporkan tidak cocok; dipakai hasil hitung


In [24]:
final = p.rename(columns={'lahan_terbakar_harian_ha': 'lahan_harus_dipadamkan_ha'})
final = final[['provinsi','lahan_harus_dipadamkan_ha','luas_lahan_ditangani_darat',
               'luas_lahan_ditangani_udara','dipadamkan_ha','sisa_ha','pct_tertangani',
               'pct_udara','pct_dari_total_sisa','flag_data']]
final = final.sort_values('pct_tertangani').reset_index(drop=True)
final.to_csv('karhutla_progres_penanganan.csv', index=False)